# Analysis of Utilized Model Capacity (Local Effective Dimension)

This notebook implements the calculations of the Local Effective Dimension (LED) according to the methodology of Abbas et al. (2021) for the trained weights of a VQC model. 

Unlike global expressibility, LED allows for measuring the actual number of degrees of freedom and information capacity that the model utilizes **locally**, exactly at the point where the optimizer stopped ($\theta^*$).

**Goal:** Compare the local model capacity for different depths (Depth 2, 4, 6) and environments (Odra / Simulator).

In [1]:
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_algorithms.gradients import ReverseEstimatorGradient
from qiskit_machine_learning.neural_networks import LocalEffectiveDimension

from qiskit_algorithms.gradients import ParamShiftEstimatorGradient

from sklearn.preprocessing import MinMaxScaler
from ucimlrepo import fetch_ucirepo

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [2]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    m = depth // 2
    if m == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (m - 1) * full + last

def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1
        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits
        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits
        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits
        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits
    return qc

def simulator_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1
        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1
        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1
        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1
        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
    return qc

class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        self.feature_map = self._create_angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])
        
        estimator = StatevectorEstimator()
        gradient = ParamShiftEstimatorGradient(estimator)

        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient
        )
        self.quantum_layer = TorchConnector(self.qnn)

    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.quantum_layer(x)

In [3]:
# Loading data to calculate the Fisher Information Matrix (DQFIM/LED)
banknote_authentication = fetch_ucirepo(id=267)
X = banknote_authentication.data.features.to_numpy()

# Feature engineering (according to your model_odra.ipynb file)
variance = X[:, 0].reshape(-1, 1)
skewness = X[:, 1].reshape(-1, 1)
interaction = variance * skewness
X_expanded = np.hstack((X, interaction))

scaler = MinMaxScaler(feature_range=(-np.pi/4, np.pi/4))
X_scaled = scaler.fit_transform(X_expanded)

# For LED analysis, we don't need the entire dataset (it is very computationally expensive).
# We select a random sample of N=100 points to sample the information space.
SAMPLE_SIZE = 100
indices = np.random.choice(len(X_scaled), SAMPLE_SIZE, replace=False)
X_sample = X_scaled[indices]

In [ ]:
# Search for all .pth files from the 30th epoch (final weights)
# (You can change the search pattern here if you want to test other epochs)
weights_base_path = "../cross_validation/Models/Weights/"
NUM_QUBITS = 5
results = []

weight_files = glob.glob(f"{weights_base_path}/**/*epoch_30_weights.pth", recursive=True)
print(f"Found {len(weight_files)} weight files for analysis.")

for file_path in weight_files:
    # Extracting parameters from the filename
    is_noise = "noise" in file_path
    match_depth = re.search(r"depth_(\d+)", file_path)
    match_fold = re.search(r"fold_(\d+)", file_path)
    
    if not match_depth or not match_fold:
        continue
        
    depth = int(match_depth.group(1))
    fold = int(match_fold.group(1))
    environment = "Odra" if "Odra" in file_path else "Simulator"
    
    # 1. DYNAMIC SELECTION OF THE APPROPRIATE ANSATZ:
    if environment == "Odra":
        current_ansatz = odra_ansatz(NUM_QUBITS, depth)
    else:
        current_ansatz = simulator_ansatz(NUM_QUBITS, depth)
        
    model = HybridModel(current_ansatz, NUM_QUBITS)
    
    # 2. Loading weights
    try:
        model.load_state_dict(torch.load(file_path, map_location=torch.device('cpu'), weights_only=True))
        model.eval()
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        continue
        
    # Extract trained weights to a numpy array
    trained_weights = model.quantum_layer.weight.detach().numpy().reshape(1, -1)
    
    print(f"Calculating LED: Env={environment}, Depth={depth}, Fold={fold}, Noise={is_noise}...")
    
    led_calculator = LocalEffectiveDimension(
        qnn=model.qnn,
        weight_samples=trained_weights,
        input_samples=X_sample
    )
    
    # Calculate LED (dataset_size normalizes the Fisher matrix)
    eff_dim = led_calculator.get_effective_dimension(dataset_size=1097)
    
    # Qiskit might return a scalar or a 1-element array depending on the version.
    # This notation is 100% safe:
    import numpy as np
    led_value = float(np.squeeze(eff_dim))
    
    num_params = trained_weights.shape[1]
    normalized_led = led_value / num_params 
    
    results.append({
        "Environment": environment,
        "Depth": depth,
        "Fold": fold,
        "Noise": is_noise,
        "Parameters": num_params,
        "LED": led_value,
        "Normalized_LED": normalized_led
    })
    
    print(f" -> LED = {led_value:.2f} / {num_params} parameters ({normalized_led:.2%})")

print("\nEnd of calculations!")

Found 60 weight files for analysis.
Calculating LED: Env=Simulator, Depth=2, Fold=5, Noise=True...
 -> LED = 3.05 / 17 parameters (17.92%)
Calculating LED: Env=Simulator, Depth=2, Fold=5, Noise=False...
 -> LED = 3.09 / 17 parameters (18.19%)
Calculating LED: Env=Simulator, Depth=2, Fold=2, Noise=False...
 -> LED = 3.02 / 17 parameters (17.74%)
Calculating LED: Env=Simulator, Depth=2, Fold=2, Noise=True...
 -> LED = 2.97 / 17 parameters (17.45%)
Calculating LED: Env=Simulator, Depth=2, Fold=3, Noise=False...
 -> LED = 3.05 / 17 parameters (17.95%)
Calculating LED: Env=Simulator, Depth=2, Fold=3, Noise=True...
 -> LED = 3.01 / 17 parameters (17.70%)
Calculating LED: Env=Simulator, Depth=2, Fold=1, Noise=True...
 -> LED = 3.01 / 17 parameters (17.72%)
Calculating LED: Env=Simulator, Depth=2, Fold=1, Noise=False...
 -> LED = 3.07 / 17 parameters (18.07%)
Calculating LED: Env=Simulator, Depth=2, Fold=4, Noise=True...
 -> LED = 3.00 / 17 parameters (17.63%)
Calculating LED: Env=Simulator, D

In [ ]:
import pandas as pd
import seaborn as sns

# Convert results to a DataFrame for easier analysis
df_results = pd.DataFrame(results)

# Filtering only for models without noise (if you want to analyze everything, remove this filter)
df_clean = df_results[df_results["Noise"] == False].copy()

plt.figure(figsize=(12, 6))

# Plot 1: Absolute Effective Dimension (LED) vs. Depth and Environment
plt.subplot(1, 2, 1)
sns.boxplot(data=df_clean, x="Depth", y="LED", hue="Environment", palette="Set2")
plt.title("Local Effective Dimension (Absolute)")
plt.ylabel("Information Capacity (LED)")
plt.xlabel("Circuit Depth")

# Plot 2: Normalized Effective Dimension
# Allows evaluating what fraction of available weights is actually doing the work
plt.subplot(1, 2, 2)
sns.boxplot(data=df_clean, x="Depth", y="Normalized_LED", hue="Environment", palette="Set2")
plt.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Max (Ideal utilization)')
plt.title("Normalized LED (% of utilized parameters)")
plt.ylabel("LED / Number of parameters")
plt.xlabel("Circuit Depth")
plt.legend()

plt.tight_layout()
plt.show()

# Aggregation of mean results
summary = df_clean.groupby(["Environment", "Depth"])[["LED", "Normalized_LED"]].mean.round(3)
print("\n--- MEAN LOCAL EFFECTIVE DIMENSION RESULTS ---")
print(summary)

KeyError: 'Noise'